In [0]:
%pip install geopandas rasterio shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 63.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping

In [0]:
import tempfile
import shutil

flight_list = dbutils.jobs.taskValues.get(
    taskKey="3_orquestator", 
    key="missing_clips", 
    default=[]
)

if not flight_list:
    dbutils.notebook.exit("No pending flights to process. Exiting gracefully.")

print(f" Received {len(flight_list)} flights for plot cropping.\n")

for flight_path in flight_list:
    print("-" * 60)
    print(f" PROCESSING FLIGHT: {flight_path}")
    
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
        
    base_dir = os.path.dirname(flight_path)
    parent_dir = os.path.dirname(base_dir)
    field_data_dir = os.path.join(parent_dir, "field_data")

    possible_ortho_names = ["RGB.tif", "MS.tif"]
    rgb_path = None
    for name in possible_ortho_names:
        candidate = os.path.join(base_dir, name)
        if os.path.exists(candidate):
            rgb_path = candidate
            break

    out_dir = os.path.join(base_dir, "plot_clipped") 
    
    print(f" Searching geometries in: {field_data_dir}")
    vector_files = glob.glob(os.path.join(field_data_dir, "*.shp")) + glob.glob(os.path.join(field_data_dir, "*.geojson"))
    
    if rgb_path is None:
        print(f" Error: No RGB.tif or MS.tif found in {base_dir}. Skipping flight...")
        continue
    else:
        print(f" Orthomosaic found: {os.path.basename(rgb_path)}")
        
    if not vector_files:
        print(f" Error: No .shp or .geojson file found in {field_data_dir}. Skipping flight...")
        continue

    vector_path = vector_files[0]
    print(f" Vector file found: {os.path.basename(vector_path)}")

    os.makedirs(out_dir, exist_ok=True)

    # Local scratch directory on the cluster's own disk (not the Volume)
    local_tmp_dir = tempfile.mkdtemp(prefix="plot_clip_")

    try:
        print(" Loading geometries and checking Coordinate Reference Systems (CRS)...")
        gdf_plots = gpd.read_file(vector_path)
        
        with rasterio.open(rgb_path) as src:
            raster_crs = src.crs
            
            if gdf_plots.crs != raster_crs:
                print(f" Reprojecting polygons from {gdf_plots.crs} to {raster_crs}...")
                gdf_plots = gdf_plots.to_crs(raster_crs)

            n_bands = src.count
            if n_bands <= 4:
                out_driver = "PNG"
                out_ext = "png"
            else:
                out_driver = "GTiff"
                out_ext = "tif"
            print(f" Detected {n_bands} bands -> using {out_driver} for output ({out_ext}).")

            print(f" Clipping {len(gdf_plots)} detected plots...")
            
            for idx, row in gdf_plots.iterrows():
                geometry = [mapping(row.geometry)]
                
                out_image, out_transform = mask(src, geometry, crop=True)
                
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": out_driver,  
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })
                
                file_name = f"Plot_{idx + 1}.{out_ext}"
                
                # Write to LOCAL disk first (avoids seek errors on the Volume)
                local_out_path = os.path.join(local_tmp_dir, file_name)
                with rasterio.open(local_out_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                
                # Then copy the finished file to the Volume
                final_out_path = os.path.join(out_dir, file_name)
                shutil.copyfile(local_out_path, final_out_path)
                    
            print(f" SUCCESS: {len(gdf_plots)} images saved to {out_dir}")

    except Exception as e:
        print(f" An error occurred processing this flight: {e}")
    
    finally:
        # Clean up local scratch files regardless of success/failure
        shutil.rmtree(local_tmp_dir, ignore_errors=True)

print("\n" + "="*70)
print(" PLOT CROPPING PIPELINE FINISHED SUCCESSFULLY.")

In [0]:
    import requests
    
    JOB_4_ID = 713354967172258
    

    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = ctx.apiUrl().get()
    token = ctx.apiToken().get()
    
    
    url = f"{host}/api/2.1/jobs/run-now"
    headers = {"Authorization": f"Bearer {token}"}
    data = {"job_id": JOB_4_ID}
    
    
    response = requests.post(url, headers=headers, json=data)
    
    if response.status_code == 200:
        print(" Trigger successful! PhenoI Job has been started.")
    else:
        print(f" Error triggering Job: {response.text}")